## 7. Preprocessing and Feature Engineering

Before training our models, the dataset needs to be prepared for the forecasting task. This involves organizing the observations chronologically, removing unnecessary attributes and creating new features from historical and temporal information.

In [65]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/raw/energydata_complete.csv')

display(df.head())
print("Dataset shape:", df.shape)

,date,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,...,T9,RH_9,T_out,Press_mm_hg,RH_out,Windspeed,Visibility,Tdewpoint,rv1,rv2
0,2016-01-11 17:00:00,60,30,19.89,47.596667,19.2,44.790000,19.79,44.730000,19.000000,...,17.033333,45.53,6.600000,733.5,92.0,7.000000,63.000000,5.3,13.275433,13.275433
1,2016-01-11 17:10:00,60,30,19.89,46.693333,19.2,44.722500,19.79,44.790000,19.000000,...,17.066667,45.56,6.483333,733.6,92.0,6.666667,59.166667,5.2,18.606195,18.606195
2,2016-01-11 17:20:00,50,30,19.89,46.300000,19.2,44.626667,19.79,44.933333,18.926667,...,17.000000,45.50,6.366667,733.7,92.0,6.333333,55.333333,5.1,28.642668,28.642668
3,2016-01-11 17:30:00,50,40,19.89,46.066667,19.2,44.590000,19.79,45.000000,18.890000,...,17.000000,45.40,6.250000,733.8,92.0,6.000000,51.500000,5.0,45.410389,45.410389
4,2016-01-11 17:40:00,60,40,19.89,46.333333,19.2,44.530000,19.79,45.000000,18.890000,...,17.000000,45.40,6.133333,733.9,92.0,5.666667,47.666667,4.9,10.084097,10.084097


Dataset shape: (19735, 29)


### 7.1. Preparing the Dataset

The initial preprocessing steps focus on converting the timestamp into the appropriate format, removing irrelevant attributes and ensuring that observations are correctly ordered in time.

In [66]:
df['date'] = pd.to_datetime(df['date'])

df = df.drop(columns=['rv1', 'rv2'])

df = df.sort_values(by='date').reset_index(drop=True)

print("Dataset shape after changes:", df.shape)

Dataset shape after changes: (19735, 27)


### 7.2. Creating the Target Variable

The objective of this project is to predict appliance energy consumption 10 minutes into the future, using information available at the current time.

Since the dataset contains measurements recorded at 10-minute intervals, we can create a new target variable by shifting the `Appliances` column one position upwards. This aligns each observation with the appliance consumption of the following time interval.

In [67]:
df['target'] = df['Appliances'].shift(-1)
display(df[['date', 'Appliances', 'target']].tail(4))

print("Since the last entry now has a NaN value, we will be removing it")
df = df.dropna(subset=['target']).reset_index(drop=True)

display(df[['date', 'Appliances', 'target']].head())

print("Dataset shape:", df.shape)
print("Missing target values:", df['target'].isnull().sum())

,date,Appliances,target
19731,2016-05-27 17:30:00,90,270.0
19732,2016-05-27 17:40:00,270,420.0
19733,2016-05-27 17:50:00,420,430.0
19734,2016-05-27 18:00:00,430,NaN


Since the last entry now has a NaN value, we will be removing it


,date,Appliances,target
0,2016-01-11 17:00:00,60,60.0
1,2016-01-11 17:10:00,60,50.0
2,2016-01-11 17:20:00,50,50.0
3,2016-01-11 17:30:00,50,60.0
4,2016-01-11 17:40:00,60,50.0


Dataset shape: (19734, 28)
Missing target values: 0


### 7.3. Creating Historical Features

Historical consumption values can provide useful information for predicting future energy consumption. We will create lag features to represent appliance consumption at different points in the past.

The selected lag intervals are based on the temporal patterns observed during EDA:

- **10, 20 and 30 minutes (lags 1, 2 and 3):** Appliance consumption showed strong autocorrelation between consecutive observations. These short-term lags allow the models to capture recent consumption patterns.
- **1 hour (lag 6):** This interval provides information about consumption over a slightly longer period and helps capture short-term changes.
- **24 hours (lag 144):** The ACF analysis showed evidence of daily patterns, with recurring peaks around multiples of 144 observations. This lag allows the models to use consumption from the same time on the previous day.

In addition to individual historical observations, we will calculate a rolling average over the previous hour to represent recent consumption trends.

In [68]:
lags = [1, 2, 3, 6, 144]

for lag in lags:
    df[f'Appliances_lag_{lag}'] = df['Appliances'].shift(lag)

df['Appliances_rolling_mean_1h'] = (
    df['Appliances'].shift(1).rolling(window=6).mean()
)

Creating these lag features introduces missing values at the beginning of the dataset. Since the largest lag is 144, the first 144 rows will contain at least one missing value. We can remove these rows.

In [69]:
lag_columns = [f'Appliances_lag_{lag}' for lag in lags]

df = df.dropna(
    subset=lag_columns + ['Appliances_rolling_mean_1h']
).reset_index(drop=True)

### Inspecting the data

In [70]:
display(df[['date', 'Appliances', 'target',
            'Appliances_lag_1', 'Appliances_lag_6',
            'Appliances_lag_144',
            'Appliances_rolling_mean_1h']].head(10))

print("Dataset shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())

,date,Appliances,target,Appliances_lag_1,Appliances_lag_6,Appliances_lag_144,Appliances_rolling_mean_1h
0,2016-01-12 17:00:00,60,60.0,40.0,30.0,60.0,40.000000
1,2016-01-12 17:10:00,60,210.0,60.0,40.0,60.0,45.000000
2,2016-01-12 17:20:00,210,380.0,60.0,50.0,50.0,48.333333
3,2016-01-12 17:30:00,380,370.0,210.0,40.0,50.0,75.000000
4,2016-01-12 17:40:00,370,230.0,380.0,40.0,60.0,131.666667
5,2016-01-12 17:50:00,230,80.0,370.0,40.0,50.0,186.666667
6,2016-01-12 18:00:00,80,90.0,230.0,60.0,60.0,218.333333
7,2016-01-12 18:10:00,90,70.0,80.0,60.0,60.0,221.666667
8,2016-01-12 18:20:00,70,40.0,90.0,210.0,60.0,226.666667
9,2016-01-12 18:30:00,40,40.0,70.0,380.0,70.0,203.333333


Dataset shape: (19590, 34)
Missing values: 0


### 7.4. Creating Temporal Features

Appliance energy consumption can vary depending on the time of day and day of the week. To capture these temporal patterns, we will extract additional features from the `date` column.

We will use the hour of the day and the day of the week as additional inputs for our models.

In [71]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.day_of_week

Since the hour of the day is a cyclical variable, representing it as an ordinary numerical value can introduce an artificial distance between 23:00 and 00:00. To preserve the circular nature of time, we will transform the hour into sine and cosine features using a 24-hour cycle.

In [72]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

### Inspecting the data

In [ ]:
display(df[['date', 'hour', 'day_of_week',
            'hour_sin', 'hour_cos']].head(10))

,date,hour,day_of_week,hour_sin,hour_cos
0,2016-01-12 17:00:00,17,1,-0.965926,-2.588190e-01
1,2016-01-12 17:10:00,17,1,-0.965926,-2.588190e-01
2,2016-01-12 17:20:00,17,1,-0.965926,-2.588190e-01
3,2016-01-12 17:30:00,17,1,-0.965926,-2.588190e-01
4,2016-01-12 17:40:00,17,1,-0.965926,-2.588190e-01
5,2016-01-12 17:50:00,17,1,-0.965926,-2.588190e-01
6,2016-01-12 18:00:00,18,1,-1.000000,-1.836970e-16
7,2016-01-12 18:10:00,18,1,-1.000000,-1.836970e-16
8,2016-01-12 18:20:00,18,1,-1.000000,-1.836970e-16
9,2016-01-12 18:30:00,18,1,-1.000000,-1.836970e-16


### 7.5. Finalizing the Feature Set

After creating the historical and temporal features, we will prepare the dataset for the modelling stage.

The `date` column will no longer be needed as a direct input, since we have extracted the temporal information required for our predictions. We will also separate the target variable from the remaining features.

In [74]:
X = df.drop(columns=['date', 'target'])
y = df['target']    

print("Features shape:", X.shape)
print("Target shape:", y.shape)

display(X.head())
display(y.head())

Features shape: (19590, 36)
Target shape: (19590,)


,Appliances,lights,T1,RH_1,T2,RH_2,T3,RH_3,T4,RH_4,...,Appliances_lag_1,Appliances_lag_2,Appliances_lag_3,Appliances_lag_6,Appliances_lag_144,Appliances_rolling_mean_1h,hour,day_of_week,hour_sin,hour_cos
0,60,0,20.066667,42.833333,19.000000,42.418182,19.790000,44.700000,19.26,42.560000,...,40.0,40.0,40.0,30.0,60.0,40.000000,17,1,-0.965926,-0.258819
1,60,10,20.000000,42.672500,19.000000,42.433333,19.790000,44.663333,19.20,42.560000,...,60.0,40.0,40.0,40.0,60.0,45.000000,17,1,-0.965926,-0.258819
2,210,20,20.000000,42.530000,18.990000,42.471818,19.790000,44.590000,19.20,42.626667,...,60.0,60.0,40.0,50.0,50.0,48.333333,17,1,-0.965926,-0.258819
3,380,20,20.033333,43.496667,18.902222,42.580000,19.823333,44.590000,19.20,42.760000,...,210.0,60.0,60.0,40.0,50.0,75.000000,17,1,-0.965926,-0.258819
4,370,40,20.033333,42.963333,18.890000,42.560000,19.890000,44.590000,19.36,43.566667,...,380.0,210.0,60.0,40.0,60.0,131.666667,17,1,-0.965926,-0.258819


0     60.0
1    210.0
2    380.0
3    370.0
4    230.0
Name: target, dtype: float64

In [75]:
print("Input features:")
print(X.columns.tolist())

Input features:
['Appliances', 'lights', 'T1', 'RH_1', 'T2', 'RH_2', 'T3', 'RH_3', 'T4', 'RH_4', 'T5', 'RH_5', 'T6', 'RH_6', 'T7', 'RH_7', 'T8', 'RH_8', 'T9', 'RH_9', 'T_out', 'Press_mm_hg', 'RH_out', 'Windspeed', 'Visibility', 'Tdewpoint', 'Appliances_lag_1', 'Appliances_lag_2', 'Appliances_lag_3', 'Appliances_lag_6', 'Appliances_lag_144', 'Appliances_rolling_mean_1h', 'hour', 'day_of_week', 'hour_sin', 'hour_cos']
